# API HH
https://api.hh.ru/openapi/redoc#tag/Poisk-vakansij/operation/get-vacancies - описание api

# ANALYZE

Что хотим получить в итоге?
1. Требования
- топ требований (например, стек технологий)
- опыт работы
- дополнителньые требования какие-то
2. Обязанности
- какие основные обязанности (собрать все обязанности, объеденить)
3. Условия труда
- Какой режим (онлайн/оффлайн/гибрид, какой график работы)
- Условия труда (что предлагают, релокейт, плюшки)
- В каком месте сколько вакансий (например, в москве 100 вакансий, в перми 20 вакансий)
- ЗП (диапазон зарплат, график распределния зп, сравнеие по регионам, средняя зп)
4. Идеальный кандидат (Собрать все данные и сделать из них "идеального кандидата", который подходит 90% вакансий.

Вот какие-то короткие данные хочется получичить в качестве сообщения от тг бота текстом. Какие-то большие данные, подробности хочется получить в pdf файле. Например, все графики можно отправить в pdf. В сообщении просто, например, идеального кандидата

In [ ]:
!pip install -U g4f[all] mysql-connector-python openpyxl matplotlib telebot

In [2]:
# Стандартные библиотеки
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from io import BytesIO
import io
import os
import random
import re
from statistics import mean, median
from threading import Lock
import xml.etree.ElementTree as ET

# Сторонние библиотеки
from bs4 import BeautifulSoup
import cachetools
from cachetools import TTLCache
import g4f
from g4f.client import Client
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import mysql.connector
from mysql.connector import Error
import openpyxl
from openpyxl import Workbook
from openpyxl.chart import BarChart, PieChart, Reference, Series
from openpyxl.drawing.image import Image
from openpyxl.styles import Border, Font, Side
from openpyxl.worksheet.table import Table, TableStyleInfo
import requests
import telebot
from telebot import types

In [3]:
# Список доступных хостов
hosts = [
    "hh.ru", "rabota.by", "hh1.az", "hh.uz", "hh.kz",
    "headhunter.ge", "headhunter.kg"
]

In [4]:
HOST_URLS = {
    "hh.ru": "https://api.hh.ru/vacancies/{}",
    "rabota.by": "https://api.rabota.by/vacancies/{}",
    "hh1.az": "https://api.hh1.az/vacancies/{}",
    "hh.uz": "https://api.hh.uz/vacancies/{}",
    "hh.kz": "https://api.hh.kz/vacancies/{}",
    "headhunter.ge": "https://api.headhunter.ge/vacancies/{}",
    "headhunter.kg": "https://api.headhunter.kg/vacancies/{}"
}

In [5]:
def get_vacancies(text, per_page = 100, host ="hh.ru"):
  url_get_vacancies = f"https://api.{host}/vacancies"

  vacancies = []
  for page in range(20):
      params = {
          "text": text,
          "host": host,
          "page": page,
          "per_page": per_page}
      try:
          response = requests.get(url_get_vacancies, params=params)
          if response.status_code == 200:
              data = response.json()
              vacancies_per_page = data.get("items", [])
              if not vacancies_per_page:
                  break
              vacancies.extend(vacancies_per_page)
          else:
              break
      except requests.RequestException as e:
              print(f"Ошибка при запросе вакансий: {e}")
              break
  return vacancies

In [6]:
def get_vacancy_by_id(vacancy_id, host ="hh.ru"):
    """
    Получение вакансии по ID с учетом выбранного хоста

    :param vacancy_id: ID вакансии
    :param host: хост (по умолчанию hh.ru)
    :return: JSON с информацией о вакансии или пустой список
    """
    if host not in HOST_URLS:
        print(f"Неподдерживаемый хост: {host}")
        return []
    # Формируем URL для конкретного хоста
    url = HOST_URLS[host].format(vacancy_id)
    params = {"host": host}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Ошибка при получении вакансии с ID {vacancy_id}: {response.status_code} {response.text}")
        return []

In [7]:
def get_full_vacancies(vacancies, host="hh.ru", limit = 100):
  """
    Получение полной информации о вакансиях

    :param vacancies: список вакансий
    :param host: хост (по умолчанию hh.ru)
    :param limit: максимальное количество вакансий для обработки
    :return: список полных вакансий
    """
  full_vacancies = []
  for v in vacancies[:min(limit, len(vacancies))]:
      vac = get_vacancy_by_id(v["id"], host)
      if not vac:
          break
      full_vacancies.append(vac)
  return full_vacancies

In [8]:
# получить ссылку на страницу
def get_url(text, host = "hh.ru"):
    # Генерация URL для поиска вакансий
    return f"https://{host}/search/vacancy?text={text}"

# БД

In [9]:
db_connection = None

In [10]:
def get_db_connection():
    return mysql.connector.connect(
        host="bvpdcrqf2ifq5cy3wovh-mysql.services.clever-cloud.com",
        user="u5o2etsggrwgfpet",
        password="TQMaDmSC0dCSGDCEOQEd",
        database="bvpdcrqf2ifq5cy3wovh"
    )

In [11]:
def log_message_to_db(message, host, vacancy_description):
    if db_connection is None:
        return
    try:
        cursor = db_connection.cursor()
        insert_query = """
        INSERT INTO telegram_messages (message_id, user_id, first_name, username, message_date, text, host)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """
        cursor.execute(
            insert_query,
            (
                message.message_id,
                message.from_user.id,
                message.from_user.first_name,
                message.from_user.username,
                datetime.utcfromtimestamp(message.date),
                vacancy_description,
                host
            ),
        )
        db_connection.commit()
    except Error as e:
        print(f"Ошибка при сохранении сообщения: {e}")

## Excel

In [12]:
def create_empty_excel():
    wb = Workbook()
    return wb

In [13]:
def save_excel_file(wb, chat_id, text, host):
    file_name = f"{chat_id}_{text[:50]}_{host[:(len(host)-3)]}.xlsx"
    file_path = f"/tmp/{file_name}"
    wb.save(file_path)
    return file_path

In [14]:
def send_excel_file(wb, chat_id, text, host):
    file_path = save_excel_file(wb, chat_id, text, host)
    with open(file_path, "rb") as file:
        bot.send_document(chat_id, file, caption="📗 Вот ваш файл с анализом!")
    os.remove(file_path)

In [15]:
def add_excel_filter(wb, headers, sz):
  ws = wb.active
  ws.auto_filter.ref = f"A1:{chr(65 + len(headers) - 1)}1"

In [16]:
def get_list_by_name(values):
  new_list = []
  for v in values:
    new_list.append(v['name'])
  return new_list

In [17]:
def get_string_list_by_name(values):
  need = get_list_by_name(values)
  if not need:
    return ""
  return "; ".join(need)

In [18]:
def set_all_borders(ws, data_range):
    thin_border = Border(
        left=Side(border_style="thin"),
        right=Side(border_style="thin"),
        top=Side(border_style="thin"),
        bottom=Side(border_style="thin")
    )

    for row in ws[data_range]:
        for cell in row:
            cell.border = thin_border

In [19]:
def print_excel_vacancies(wb, vacancies, host = "hh.ru"):
  ws = wb.active
  ws.title = "Вакансии"
  if not vacancies:
    return

  headers = ["ID вакансии", "Вакансия", "Город", "График работы", "Опыт работы",
             "Занятость", "Ночные смены", "Стажировка", "Рабочие часы",
             "Рабочие смены", "Профессиональные роли", "ЗП от", "ЗП до", "Валюта",
             "Тип зарплаты"]
  ws.append(headers)
  for cell in ws[1]:
        cell.font = Font(bold=True)

  for vacancy in vacancies:
        vacancy_url = f"https://{host}/vacancy/{vacancy['id']}"

        id = vacancy['id']
        name = vacancy['name']
        area = vacancy['area']['name']
        graphic_work = vacancy['schedule']['name']
        experience = vacancy['experience']['name']
        employment = vacancy['employment']['name']

        night_shifts = "Есть" if vacancy['night_shifts'] else "Нет"
        internship = "Есть" if vacancy['internship'] else "Нет"

        working_hours = get_string_list_by_name(vacancy['working_hours'])
        work_schedule_by_days = get_string_list_by_name(vacancy['work_schedule_by_days'])
        professional_roles = get_string_list_by_name(vacancy['professional_roles'])

        salary_from = ""
        salary_to = ""
        currency = ""
        gross = ""
        if vacancy['salary'] is not None:
          salary_from = vacancy['salary']['from'] if vacancy['salary']['from'] is not None else ""
          salary_to = vacancy['salary']['to'] if vacancy['salary']['to'] is not None else ""
          currency = vacancy['salary']['currency']
          gross = "До вычета" if vacancy['salary']['gross'] else "На руки"

        ws.append([f'=HYPERLINK("{vacancy_url}", "{id}")',
           name,
           area,
           graphic_work,
           experience,
           employment,
           night_shifts,
           internship,
           working_hours,
           work_schedule_by_days,
           professional_roles,
           salary_from,
           salary_to,
           currency,
           gross
           ])
  rows_count = len(vacancies) + 1
  columns_count = len(headers)
  data_range = f"A1:{chr(65 + columns_count - 1)}{rows_count}"
  set_all_borders(ws, data_range)
  add_excel_filter(wb, headers, len(vacancies))

In [20]:
def add_pie_chart(ws, sorted_dict, chart_title):
    # Создаем pie chart
    pie = PieChart()
    pie.title = chart_title
    labels = Reference(ws, min_row=2, max_row=len(sorted_dict)+1, min_col=1)
    data = Reference(ws, min_row=2, max_row=len(sorted_dict)+1, min_col=2)
    pie.add_data(data, titles_from_data=False)
    pie.set_categories(labels)

    # Размещаем график справа от данных
    ws.add_chart(pie, f"D2")

In [21]:
def add_bar_chart(ws, sorted_dict, chart_title):
    # Создаем bar chart
    bar = BarChart()
    bar.type = "col"
    bar.title = chart_title
    labels = Reference(ws, min_row=2, max_row=len(sorted_dict)+1, min_col=1)
    data = Reference(ws, min_row=2, max_row=len(sorted_dict)+1, min_col=2)
    bar.add_data(data, titles_from_data=False)
    bar.set_categories(labels)

    # Размещаем график справа от данных
    ws.add_chart(bar, f"D2")

In [22]:
def print_excel_dict(wb, sorted_dict, name, chart_type='pie'):
    ws = wb.create_sheet(name)
    wb.active = ws
    headers = [name, "Количество"]
    ws.append(headers)
    for cell in ws[1]:
        cell.font = Font(bold=True)

    for val, c in sorted_dict:
        ws.append([val, c])

    rows_count = len(sorted_dict) + 1
    columns_count = len(headers)
    data_range = f"A1:{chr(65 + columns_count - 1)}{rows_count}"
    set_all_borders(ws, data_range)

    # Добавляем график
    if chart_type == 'pie':
        add_pie_chart(ws, sorted_dict, f"Распределение по {name}")
    elif chart_type == 'bar':
        add_bar_chart(ws, sorted_dict, f"Распределение по {name}")

In [23]:
def print_excel_salary(wb, salary_data):
    ws = wb.create_sheet("Зарплата")
    wb.active = ws

    # Добавляем заголовки
    headers = ["Метрика", "Значение"]
    ws.append(headers)
    for cell in ws[1]:
        cell.font = Font(bold=True)

    # Добавляем данные о зарплате
    salary_metrics = [
        ("Средняя зарплата", salary_data['avg_salary']),
        ("Медианная зарплата", salary_data['median_salary']),
        ("Минимальная зарплата", salary_data['min_salary']),
        ("Максимальная зарплата", salary_data['max_salary'])
    ]

    for metric, value in salary_metrics:
        ws.append([metric, value])

    # Создаем bar chart для распределения зарплат
    if salary_data.get('salary_ranges'):
        ranges_data = sorted(salary_data['salary_ranges'].items())

        # Добавляем данные о диапазонах зарплат
        ws.append([])  # Пустая строка
        ws.append(["Диапазон зарплат", "Количество вакансий"])
        for val, c in ranges_data:
            ws.append([val, c])

        # Добавляем bar chart для диапазонов зарплат
        add_bar_chart(ws, ranges_data, "Распределение вакансий по зарплатным диапазонам")

## Полезные функции для анализа

In [24]:
def get_frequency_dict_by_name(vacancies, field):
  frequency_dict = {}
  for v in vacancies:
    frequency_dict[v[field]["name"]] = frequency_dict.get(v[field]["name"], 0) + 1
  return frequency_dict

In [25]:
def get_frequency_dict_by_field(vacancies, field):
    frequency_dict = {"Есть":0, "Нет":0}
    for v in vacancies:
      if v[field]:
        frequency_dict["Есть"] =frequency_dict.get("Есть", 0) + 1
      else:
        frequency_dict["Нет"] =frequency_dict.get("Нет", 0) + 1
    return frequency_dict

In [26]:
def get_frequency_dict_by_list_field(vacancies, field):
  frequency_dict = {}
  for v in vacancies:
    if v[field]:
      for value in v[field]:
         frequency_dict[value["name"]] = frequency_dict.get(value["name"], 0) + 1
  return frequency_dict

In [27]:
def get_sorted_frequenct(vacancies, field, dict_to_sort):
  sorted_frequency = sorted(dict_to_sort.items(), key=lambda item: item[1], reverse=True)
  return sorted_frequency

In [28]:
def print_top(chat_id, sz, dict_list, msg):
  for i in range(sz):
    name, count = dict_list[i]
    msg += f"{name}: {count}\n"
  bot.send_message(chat_id, msg)

In [29]:
def process_field_name(wb, chat_id, vacancies, field, symbol = "", top = 10):
  freq = get_frequency_dict_by_name(vacancies, field)
  sorted_frequency = get_sorted_frequenct(vacancies, field, freq)
  print_excel_dict(wb, sorted_frequency, field)

  msg = f"{symbol}\n"
  if (top < len(sorted_frequency)):
    msg += f"🏆 Топ {top} по количеству вакансий:\n"

  top = min(top, len(sorted_frequency))
  print_top(chat_id, top, sorted_frequency, msg)

In [30]:
def process_binary_field(wb, chat_id, vacancies, field, symbol = ""):
  freq = get_frequency_dict_by_field(vacancies, field)
  sorted_frequency = get_sorted_frequenct(vacancies, field, freq)
  print_excel_dict(wb, sorted_frequency, field)

  msg = f"{symbol}\n"
  print_top(chat_id, len(sorted_frequency), sorted_frequency, msg)

In [31]:
def process_list_field(wb, chat_id, vacancies, field, symbol = "", top = 5):
  freq = get_frequency_dict_by_list_field(vacancies, field)
  sorted_frequency = get_sorted_frequenct(vacancies, field, freq)
  print_excel_dict(wb, sorted_frequency, field)

  msg = f"{symbol}\n"
  top = min(top, len(sorted_frequency))
  print_top(chat_id, top, sorted_frequency, msg)

## Обработка полных вакансий
https://github.com/Tabintaban/g4f?ysclid=mm2gmiw72o96005810
1. Description
2. key_skills

In [32]:
def ask_gpt(messages, max_tokens=2000):
    providers = [
        (g4f.Provider.PollinationsAI, "openai"),
        (g4f.Provider.PollinationsAI, "openai-large"),  # запасной
    ]
    client = Client()
    for provider, model in providers:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=max_tokens,
                provider=provider
            )
            result = response.choices[0].message.content
            if result:
                return result
        except Exception as e:
            print(f"{provider.__name__} / {model}: {e}")
    return None

In [33]:
def send_long_message(chat_id, text, max_length=4096):
    if len(text) <= max_length:
        bot.send_message(chat_id, text)
        return
    while text:
        if len(text) <= max_length:
            bot.send_message(chat_id, text)
            break
        split_pos = text.rfind('\n', 0, max_length)
        if split_pos == -1:
            split_pos = max_length
        bot.send_message(chat_id, text[:split_pos])
        text = text[split_pos:].lstrip('\n')

In [34]:
def clean_html_description(combined_description):
    # Удаление HTML-тегов
    soup = BeautifulSoup(combined_description, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)

    # Удаление лишних пробелов и переносов строк
    text = re.sub(r'\s+', ' ', text).strip()

    # Удаление специальных символов
    text = re.sub(r'[^a-zA-Zа-яА-Я0-9\s\.\,\!\?]', '', text)

    return text

In [35]:
def process_description(chat_id, full_vacancies, text):

  combined_descriptions = " ".join([vacancy["description"] for vacancy in full_vacancies])
  cleaned_description = clean_html_description(combined_descriptions)
  cleaned_description = cleaned_description[:20000] # примерно 5000 токенов

  # Формирование расширенного промпта для создания сводного профиля
  messages = [
      {
          "role": "system",
          "content": """Ты эксперт по подбору персонала и анализу резюме.
          Твоя задача - создать максимально детальный профиль идеального кандидата,
          который объединяет лучшие качества и навыки из предоставленных описаний.

          Алгоритм анализа:
          1. Провести глубокий сравнительный анализ всех представленных профилей
          2. Выделить уникальные и ключевые компетенции
          3. Синтезировать идеальный профиль, который превосходит каждый из представленных

          При составлении профиля учти:
          - Технические навыки и компетенции
          - Профессиональный опыт и достижения
          - Личностные и soft skills
          - Образование и дополнительное обучение
          - Потенциал развития и карьерные амбиции

          Структура профиля:
          🏆 Сводный технический профиль
          🌟 Интегрированные профессиональные компетенции
          👥 Комплексный личностный портрет
          🚀 Стратегия профессионального роста"""
      },
      {
          "role": "user",
          "content": f"""Проанализируй описания кандидатов и создай максимально
          совершенный профиль, который объединяет лучшие характеристики.

          Описания кандидатов:
          {cleaned_description}

          Задача: Создать эталонный профиль, который превосходит
          каждый из представленных по ключевым параметрам.
          Также не пиши лишние вводные и поясняющие слова, только описание
          Если не получается обобщить, то напиши сама логично подходящие для вакансии {text} пункты профиля идеального кандидата"""
      }
  ]

  try:
      # Извлечение сгенерированного текста
      ideal_candidate_profile = ask_gpt(messages)
      if not ideal_candidate_profile:
          bot.send_message(chat_id, "Не удалось сгенерировать профиль кандидата.")
          return

      ideal_candidate_profiles[chat_id] = ideal_candidate_profile
      send_long_message(chat_id, f"🏅 Профиль идеального кандидата:\n{ideal_candidate_profile}")

  except Exception as e:
      print(f"Ошибка при генерации профиля: {e}")
      bot.send_message(chat_id, f"Извините, произошла ошибка при генерации профиля идеального кандидата. Попробуйте позже")


In [36]:
def process_full_vacancies(wb, chat_id, full_vacancies, text = ""):
    msg = "✨ Ключевые навыки:"
    process_list_field(wb, chat_id, full_vacancies, "key_skills", msg, top=10)
    process_description(chat_id, full_vacancies, text)

## Резюме

In [37]:
# ideal_candidate_profile берётся из глобального словаря если есть
def analyze_resume(chat_id, resume_text):
    ideal_candidate_profile = ideal_candidate_profiles.get(chat_id, "")

    if ideal_candidate_profile:
        # Сравнение с профилем идеального кандидата
        system_content = """Ты высококвалифицированный HR-аналитик с глубоким пониманием рынка труда.
Твоя задача - провести детальный сравнительный анализ резюме кандидата с профилем идеального кандидата.

Формат ответа:
🎯 Шанс получения оффера: X%
✅ Сильные соответствия профилю:
- Список ключевых совпадений
🔍 Расхождения с идеальным профилем:
- Конкретные недостающие компетенции
🚀 Рекомендации по развитию:
- Стратегия закрытия gaps"""
        user_content = f"""Проведи детальный сравнительный анализ:

Профиль идеального кандидата:
{ideal_candidate_profile}

Резюме кандидата:
{resume_text}

Задача: Максимально точно оценить соответствие кандидата требованиям рынка."""
    else:
        # Самостоятельный анализ без профиля
        system_content = """Ты профессиональный HR-эксперт и рекрутер с многолетним опытом.
Твоя задача - провести глубокий и объективный анализ резюме кандидата.

Формат ответа:
🎯 Шанс получения оффера: X%
📊 Сильные стороны:
- Список ключевых преимуществ
🔧 Области развития:
- Конкретные рекомендации
💡 Стратегические советы:
- Краткие рекомендации по повышению конкурентоспособности"""
        user_content = f"""Проведи детальный анализ следующего резюме:

{resume_text}

Прошу дать максимально честную и конструктивную оценку."""

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user",   "content": user_content}
    ]
    try:
        resume_analysis = ask_gpt(messages)
        if not resume_analysis:
            bot.send_message(chat_id, "Не удалось проанализировать резюме. Попробуйте позже.")
            return
        send_long_message(chat_id, f"📋 Анализ вашего резюме:\n\n{resume_analysis}")
    except Exception as e:
        print(f"Ошибка при анализе резюме: {e}")
        bot.send_message(chat_id, "Извините, произошла ошибка при анализе резюме. Попробуйте позже.")

## Зарплата

In [38]:
class CurrencyConverter:
    def __init__(self, ttl=43.200): #12 часов
        self.cache = TTLCache(maxsize=100, ttl=ttl)
        self.lock = Lock()

    def fetch_exchange_rates(self):
        today = datetime.now().strftime("%d/%m/%Y")
        url = f"https://cbr.ru/scripts/XML_daily.asp?date_req={today}"
        with self.lock:  # Блокируем доступ для других потоков
            response = requests.get(url)
            if response.status_code != 200:
                raise ValueError("Не удалось получить данные о курсах валют.")

            tree = ET.fromstring(response.content)
            rates = {"RUR": 1.0}  # RUR всегда равен 1.0
            for valute in tree.findall("Valute"):
                char_code = valute.find("CharCode").text
                value = float(valute.find("Value").text.replace(",", "."))
                nominal = int(valute.find("Nominal").text)
                rates[char_code] = value / nominal
            rates["BYR"] = rates["BYN"]
            self.cache.update(rates)

    def get_rate(self, currency):
        if currency not in self.cache:
            self.fetch_exchange_rates()
        return self.cache.get(currency, None)

In [39]:
converter = CurrencyConverter()

In [40]:
def get_converted_salaries(salaries):
  converted_salaries = []
  for salary in salaries:
    currency = salary.get("currency")
    salary_from = salary.get("from")
    salary_to = salary.get("to")

    if currency is None:
      continue

    if currency == "RUR":
      converted_salaries.append(salary)
      continue

    rate = converter.get_rate(currency)
    if rate is None:
        continue

    if salary_from:
        salary_from = salary_from * rate
    if salary_to:
        salary_to = salary_to * rate

    converted_salaries.append({
        "from": salary_from,
        "to": salary_to,
        "currency": "RUR",
        "gross": salary.get("gross")
    })
  return converted_salaries

In [41]:
def process_specific_salary(chat_id, salaries, text = ""):
  converted_salaries = get_converted_salaries(salaries)
  msg = ""
  if text:
    msg += text
  msg += f"Получено вакансий: {len(converted_salaries)}\n"

  from_values = [s["from"] for s in salaries if s["from"] is not None]
  to_values = [s["to"] for s in salaries if s["to"] is not None]

  min_from = int(round(min(from_values, default=0))) if from_values else None
  max_to = int(round(max(to_values, default=0))) if to_values else None

  mean_from = int(round(mean(from_values))) if from_values else None
  mean_to = int(round(mean(to_values))) if to_values else None

  median_from = int(round(median(from_values))) if from_values else None
  median_to = int(round(median(to_values))) if to_values else None

  avg_min_max = int(round((min_from + max_to) / 2)) if min_from is not None and max_to is not None else None
  avg_mean = int(round((mean_from + mean_to) / 2)) if mean_from is not None and mean_to is not None else None
  avg_meadian = int(round((median_from + median_to) / 2)) if median_from is not None and median_to is not None else None

  if min_from:
      msg += f"Минимальное значение 'от': {min_from:,} ₽\n"
  if max_to:
      msg += f"Максимальное значение 'до': {max_to:,} ₽\n"
  if mean_from:
      msg += f"Среднее значение 'от': {mean_from:,} ₽\n"
  if mean_to:
      msg += f"Среднее значение 'до': {mean_to:,} ₽\n"
  if median_from:
      msg += f"Медиана 'от': {median_from:,} ₽\n"
  if median_to:
      msg += f"Медиана 'до': {median_to:,} ₽\n"

  if avg_mean:
      msg += f"Средняя зарплата: {avg_mean:,} ₽\n"

  bot.send_message(chat_id, msg)

In [42]:
def process_salary(chat_id, vacancies):
    gross_salary = []
    not_gross_salary = []
    for v in vacancies:
      if v["salary"]:
        if v["salary"]["gross"]:
           gross_salary.append(v["salary"])
        else:
          not_gross_salary.append(v["salary"])

    if gross_salary:
      process_specific_salary(chat_id, gross_salary, "💄👛💍 Анализируем зарплаты до вычета налогов.\n")
    else:
      bot.send_message(chat_id, "Нет вакансий с указаной зарплатов до вычета налогов.")

    if not_gross_salary:
      process_specific_salary(chat_id, not_gross_salary, "💸💵💰 Анализируем зарплаты на руки.\n")
    else:
      bot.send_message(chat_id, "Нет вакансий с указаной зарплатов на руки.")

## Построение частотного словаря для объектов типа "object": {"id", "name"} по name
1. Города.
2. График работы.
3. Опыт работы.
4. Занятость.

In [43]:
def process_by_name(wb, chat_id, vacancies):
  fields = [
      {
          "field": "area",
          "msg" : "🌃 Города:"
       },
      {
          "field": "schedule",
          "msg" : "📊 График работы:"
       },
      {
          "field": "experience",
          "msg" : "📚 Опыт работы:"
       },
      {
          "field": "employment",
          "msg" : "🔔 Занятость:"
       }
      ]
  for field_data in fields:
    field = field_data["field"]
    msg = field_data["msg"]
    process_field_name(wb, chat_id, vacancies, field, msg)

## Построение частотного словаря для объектов типа "object": true/false
1. Ночные смены
2. Стажировка

In [44]:
def process_binary_fields(wb, chat_id, vacancies):
  fields = [
      {
          "field": "night_shifts",
          "msg" : "🌙 Ночные смены:"
       },
      {
          "field": "internship",
          "msg" : "👨‍🏫 Стажировка:"
       }
      ]
  for field_data in fields:
    field = field_data["field"]
    msg = field_data["msg"]
    process_binary_field(wb, chat_id, vacancies, field, msg)

## Построение частотного словаря для объектов типа "object": [{id, name}]
1. Рабочие часы.
2. Рабочие смены.
3. Профессиональные роли.

In [45]:
def process_list_fields(wb, chat_id, vacancies):
  fields = [
      {
          "field": "working_hours",
          "msg" : "⏱ Рабочие часы:"
       },
      {
          "field": "work_schedule_by_days",
          "msg" : "💼 Рабочие смены:"
       },
      {
          "field": "professional_roles",
          "msg" : "🛠 Профессиональные роли:"
       }
      ]
  for field_data in fields:
    field = field_data["field"]
    msg = field_data["msg"]
    process_list_field(wb, chat_id, vacancies, field, msg)

## Запуск анализа

In [46]:
user_states = {}
ideal_candidate_profiles = {}

In [47]:
# Глобальный словарь для хранения текущего описания вакансии
user_vacancy_context = {}

In [48]:
def perform_analysis(chat_id, vacancies, text, host = "hh.ru"):
  """
    Выполнение анализа вакансий с учетом выбранного хоста

    :param chat_id: ID чата
    :param vacancies: список вакансий
    :param text: текстовое описание
    :param host: хост (по умолчанию hh.ru)
    """
  if not vacancies:
    bot.send_message(
            chat_id,
            "К сожалению, по вашему описанию не найдено ни одной вакансии. 🤔\n"
            "Попробуйте переписать описание вакансии или попробуйте позже — возможно, на сервере временные проблемы."
        )
    return

  wb = openpyxl.Workbook()
  print_excel_vacancies(wb, vacancies, host)

  bot.send_message(chat_id, f"Получено вакансий: {len(vacancies)}")
  bot.send_message(chat_id, "Начинаем анализ найденных вакансий... 🕵️‍♂️")
  with ThreadPoolExecutor() as executor:
      future = executor.submit(get_full_vacancies, vacancies, host)

      process_by_name(wb, chat_id, vacancies)
      process_binary_fields(wb, chat_id, vacancies)
      process_list_fields(wb, chat_id, vacancies)

      process_salary(chat_id, vacancies)

      send_excel_file(wb, chat_id, text, host)
      full_vacancies = future.result()

  process_full_vacancies(wb, chat_id, full_vacancies, text)
  bot.send_message(chat_id, f"Анализ завершен! ✅")

  # Добавляем кнопку для анализа резюме
  markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
  markup.add(types.KeyboardButton("Отправить резюме для анализа"))

  bot.send_message(
        chat_id,
        "Хотите проверить шансы получения оффера? 🤔\n"
        "Нажмите кнопку и отправьте свое резюме в текстовом формате.",
        reply_markup=markup
    )

In [49]:
def get_ideal_candidate_profile(chat_id):
    return ideal_candidate_profiles.get(chat_id, "")

# TELEGRAM BOT

In [ ]:
bot = telebot.TeleBot("ВАШ_ТОКЕН")

In [51]:
@bot.message_handler(commands=['start'])
def start(message):
    bot.reply_to(message, "Привет! Напиши описание вакансии для анализа:")

In [52]:
@bot.message_handler(func=lambda message: True)
def handle_all_messages(message):
    chat_id = message.chat.id
    text = message.text

    # 1. Пользователь нажал кнопку «Отправить резюме для анализа»
    if text == "Отправить резюме для анализа":
        user_states[chat_id] = 'waiting_for_resume'
        bot.send_message(
            chat_id,
            "Пожалуйста, отправьте ваше резюме в текстовом формате. 📝\n"
            "Мы проведём детальный анализ ваших шансов на получение оффера.",
            reply_markup=types.ReplyKeyboardRemove()
        )
        return

    # 2. Ожидаем резюме от пользователя
    if user_states.get(chat_id) == 'waiting_for_resume':
        user_states[chat_id] = None
        analyze_resume(chat_id, text)
        bot.send_message(
            chat_id,
            "Анализ резюме завершён. Введите новое описание вакансии для нового поиска:",
            reply_markup=types.ReplyKeyboardRemove()
        )
        return

    # 3. Обычное сообщение - описание вакансии
    user_vacancy_context[chat_id] = {'description': text}

    markup = types.ReplyKeyboardMarkup(row_width=2, resize_keyboard=True)
    markup.add(*[types.KeyboardButton(host) for host in hosts])
    bot.send_message(
        chat_id,
        f"Получено описание вакансии: {text}\n\nВыберите сайт для поиска:",
        reply_markup=markup
    )
    bot.register_next_step_handler(message, process_host_selection)

In [53]:
def process_host_selection(message):
    chat_id = message.chat.id
    selected_host = message.text

    if selected_host not in hosts:
        bot.send_message(chat_id, "Пожалуйста, выберите хост из предложенных.")
        bot.register_next_step_handler(message, process_host_selection)
        return

    vacancy_description = user_vacancy_context.get(chat_id, {}).get('description')
    if not vacancy_description:
        bot.send_message(chat_id, "Произошла ошибка. Пожалуйста, начните заново с /start")
        return

    bot.send_message(
        chat_id,
        f"Вы выбрали сайт {selected_host}. Начинаем поиск...",
        reply_markup=types.ReplyKeyboardRemove()
    )

    try:
        vac = get_vacancies(vacancy_description, host=selected_host)
        if not vac:
            bot.send_message(chat_id, "К сожалению, вакансии не найдены.")
            return

        log_message_to_db(message, selected_host, vacancy_description)
        perform_analysis(chat_id, vac, vacancy_description, host=selected_host)

        url = get_url(vacancy_description, selected_host)
        bot.send_message(
            chat_id,
            f"Вы можете посмотреть вакансии по ссылке: [Кликните здесь]({url})",
            parse_mode='Markdown'
        )
    except Exception as e:
        bot.send_message(chat_id, f"Произошла ошибка при поиске: {str(e)}")
    finally:
        user_vacancy_context.pop(chat_id, None)

In [54]:
db_connection = mysql.connector.connect(
        host="bvpdcrqf2ifq5cy3wovh-mysql.services.clever-cloud.com",
        user="u5o2etsggrwgfpet",
        password="TQMaDmSC0dCSGDCEOQEd",
        database="bvpdcrqf2ifq5cy3wovh"
    )

if db_connection.is_connected():
        print("Подключение к базе данных успешно установлено.")

Подключение к базе данных успешно установлено.


In [ ]:
bot.polling()

In [58]:
if db_connection and db_connection.is_connected():
        db_connection.close()